In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
import datetime
import pickle

In [ ]:
if os.getcwd().endswith('/notebooks/figures'):
    os.chdir('../../')

### Supporting functions

In [ ]:
def hour_rounder(ts):
    # Rounds to nearest hour by adding a timedelta hour if minute >= 30
    return (ts.replace(second=0, microsecond=0, minute=0, hour=ts.hour) + datetime.timedelta(hours=ts.minute//30))

In [ ]:
def drop_nan_rows(df):
    '''Drop rows that have a NaN entry in them. Modifies dataframe inplace.'''
    
    df.dropna(inplace=True)
    return
    

def drop_missingsignal_offchannel_rows(df, feature_list, chdets_cols):
    '''Drop rows in dataframe for which ECoG signal was missing or some channel(s) were Off. 
    If ECoG signal is missing, then PLV features are 0. Modifies dataframe inplace.'''
    
    sigmiss_idx = df.index[(df[feature_list] == 0).all(axis=1)]  # if signal is missing, then PLV = 0 for all the features    
    df.drop(index=sigmiss_idx, inplace=True) 
    
    chanoff_idx = df.index[(df[chdets_cols]=='Off').any(axis=1)]  # if any channel is Off
    df.drop(index=chanoff_idx, inplace=True) # inplace
    return

def drop_lessretained_rows(df, retain_thresh):
    '''Drop rows in dataframe for which retained timepoints after removing
     stimulation artifact and buffer is < threshold. Threshold should be a fraction.
     Modifies dataframe inplace.'''
    
    df.drop(index=df.index[df['frac_timepoints_retained'] < retain_thresh], inplace=True)
    return


def include_round_timestamp(df): 
    '''Round the time to the nearest hour and include rounding error. Do this for raw and local timestamps.
    Modifies dataframe inplace.'''
    
    df['Round UTC timestamp'] = df['Raw UTC timestamp'].apply(lambda x: hour_rounder(datetime.datetime.fromisoformat(x)))
    df['Round local timestamp'] = df['Raw local timestamp'].apply(lambda x: hour_rounder(datetime.datetime.fromisoformat(x)))

    # checking
    df['Round_Raw_UTC_diff'] = df['Round UTC timestamp'] - df['Raw UTC timestamp'].apply(lambda x: datetime.datetime.fromisoformat(x))
    df['Round_Raw_UTC_diff'] = df['Round_Raw_UTC_diff'].apply(lambda x: x.total_seconds()/60) # seconds to minutes

    df['Round_Raw_Local_diff'] = df['Round local timestamp'] - df['Raw local timestamp'].apply(lambda x: datetime.datetime.fromisoformat(x))
    df['Round_Raw_Local_diff'] = df['Round_Raw_Local_diff'].apply(lambda x: x.total_seconds()/60) # seconds to minutes
    return 


def remove_duplicates_except_closest(df):
    '''Remove duplicate entries based on rounded timestamp. Keep the recordings closest to the rounded time.
    Modifies dataframe inplace.'''
    
    dupdf = df.loc[df['Round UTC timestamp'].duplicated(keep=False), 
                   ['Raw UTC timestamp','Round UTC timestamp','Round_Raw_UTC_diff']].copy()

    if dupdf.shape[0] > 0:
        
        dup_pairs_list = dupdf.groupby(by='Round UTC timestamp').apply(lambda x: tuple(x.index)).tolist() # pairs/groups of indices that are duplicated        
        drop_dup_index_list = []
        for pair in dup_pairs_list:
            keepidx = dupdf.loc[list(pair), 'Round_Raw_UTC_diff'].abs().idxmin() # one closest to actual time
            drop_dup_index_list.extend(pair) # add indices to be removed from catalog df
            drop_dup_index_list.remove(keepidx) # don't remove closest one from catalog df

        df.drop(index=drop_dup_index_list, inplace=True) # drop duplicate indices
    
    return dupdf

### Processing

In [ ]:
patient_list = ['NPE%04d'%(ii+1) for ii in range(140)]

featcsv_path = 'data/processed/ecogfeatures/pib/%s_trigger-Scheduled.csv'
save_featcsv_path = 'data/processed/ecogfeatures/pib_postproc/%s_trigger-Scheduled.csv'
save_flag = False

In [ ]:
chprefs = ['Ch_1', 'Ch_2', 'Ch_3', 'Ch_4']
bands = ['theta', 'alpha', 'beta', 'gamma', 'broadband']
feature_list = [f'{chp}_{bn}' for bn in bands for chp in chprefs]

In [ ]:
print(feature_list)

In [ ]:
chdets_cols = ['Ch 1 enabled', 'Ch 2 enabled', 'Ch 3 enabled', 'Ch 4 enabled',]

RETAIN_THRESHOLD = 0.33 # 33% of data 

for patient_id in patient_list:

    if not os.path.exists(featcsv_path % (patient_id)):
        continue

    tdf = pd.read_csv(featcsv_path % (patient_id))
    ogsize = tdf.shape[0]

    ### remove rows with any nans
    drop_nan_rows(tdf) # inplace operation
    postnansize = tdf.shape[0]

    # checks
    if postnansize < ogsize:
        print(f'{patient_id}: WARN Dropped nans')
    
    ### remove rows where signal is missing (PLV=0) or some channels are Off    
    drop_missingsignal_offchannel_rows(tdf, feature_list, chdets_cols)
    postoffsize = tdf.shape[0]
    
    if postoffsize < postnansize:
        print(f'{patient_id}: WARN Dropped {postnansize - postoffsize} missing ECoG / Off timestamps')
    
    ### remove if signal length is less than certain threshold after removing stimulation artifact
    drop_lessretained_rows(tdf, RETAIN_THRESHOLD)
    postretsize = tdf.shape[0]
    
    # checks
    if postretsize < postoffsize:
        print(f'{patient_id}: WARN Dropped {postoffsize - postretsize} since few timepoints retained')
    
    if tdf['frac_timepoints_retained'].min() < RETAIN_THRESHOLD:
        print(f'{patient_id}: WARN Minimum retained threshold breached. Aborting!')
        break
        
    ### round timestamp to the nearest hour    
    include_round_timestamp(tdf)
    
    # checks
    if (tdf['Round_Raw_UTC_diff'].abs().max() > 30) or (tdf['Round_Raw_Local_diff'].abs().max() > 30):
        print(f'{patient_id}: Has rounding diff > 30 minutes. Aborting!')
        break

    ### keep the timestamp closest to the true timestamp among the duplicate entries
    duplidf = remove_duplicates_except_closest(tdf)
    
    if duplidf.shape[0] > 0:
        print(f'{patient_id}: WARN # duplicates = {duplidf.shape[0]}')
        display(duplidf)
    
    # checks 
    if tdf.shape[0] < postretsize:
        print(f'{patient_id}: WARN Dropped {postretsize - tdf.shape[0]} duplicates.')
    
    print(f'{patient_id}: Processing complete.')
    if save_flag:
        tdf.to_csv(save_featcsv_path % (patient_id), index=False)